# Host-telemetry pipeline (Security-Datasets)

Fresh pipeline cells adapted from the CIC-IDS notebook, same function names and conventions. Run the port cell first, then A→G in order.

**Read the eval note in CELL F:** on an atomic dataset report the malicious-process result, not blanket accuracy.

In [1]:
# =====================================================================
#  PIPELINE CELLS for Security-Datasets (host telemetry)
#  Adapted from the CIC-IDS pipeline. SAME function names / conventions
#  so the comparison stays controlled. Differences are flagged inline.
#
#  Run order:  (port cell)  ->  CELL A  ->  CELL B  ->  CELL C
#              ->  CELL D  ->  CELL E  ->  CELL F  ->  CELL G
#
#  Prereqs already in your env from the CIC-IDS notebook:
#    SentenceTransformer, chromadb, networkx, llama_cpp (Llama, LlamaGrammar),
#    attck_techniques dict, MODEL_PATH, RANDOM_SEED, RESULTS_DIR, CHROMA_DIR
# =====================================================================

In [2]:
# =====================================================================
#  CELL A — EMBED + COMMUNITY DETECTION  (host alerts)
#  ---------------------------------------------------------------------
#  CHANGE vs CIC-IDS: the atomic dataset has only ~8 alerts, far too few
#  for util.community_detection to form real clusters. We therefore lower
#  min_community_size and FALL BACK to "one community = one process" when
#  clustering is degenerate. On APT29 (hundreds of alerts) remove the
#  fallback and use clustering as taught. Treat community detection here
#  as "verified runs", not as a tuned result.
# =====================================================================
from sentence_transformers import SentenceTransformer, util
import numpy as np, torch

embedder = SentenceTransformer('all-MiniLM-L6-v2')

alert_texts_all = alerts['semantic_alert'].tolist()
embeddings = embedder.encode(alert_texts_all, convert_to_tensor=True,
                             show_progress_bar=False)

N = len(alerts)
MIN_COMMUNITY_SIZE = 2 if N < 30 else 3      # relax for tiny atomic sets
communities = util.community_detection(
    embeddings, min_community_size=MIN_COMMUNITY_SIZE, threshold=0.55)

comm_ids = [-1] * N
for cid, members in enumerate(communities):
    for idx in members:
        comm_ids[idx] = cid

# Fallback: if clustering collapsed everything to noise, treat each alert
# as its own singleton community so downstream stages still execute.
if all(c == -1 for c in comm_ids):
    print("Clustering degenerate at this scale — using per-process singletons.")
    comm_ids = list(range(N))

alerts['community_id'] = comm_ids
assigned = (alerts['community_id'] != -1).sum()
print(f"{assigned}/{N} alerts assigned to {len(set(c for c in comm_ids if c!=-1))} communities")
print(alerts['community_id'].value_counts(dropna=False).head(10))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NameError: name 'alerts' is not defined

In [ ]:
# =====================================================================
#  CELL B — EXTEND RELATION VOCAB so this technique is REACHABLE
#  ---------------------------------------------------------------------
#  Your CIC-IDS SECURITY_RELATIONS already had ACCESS_CREDENTIALS, but it
#  mapped only to T1555/T1078/T1110. We add the host-relevant relations and
#  point credential-dumping at T1003 / T1003.001 so retrieval can hit the
#  ground truth. Add new relations to the ENUM, the GUIDE, the tactic map,
#  and the technique map — all four, or the grammar rejects them.
# =====================================================================
HOST_RELATIONS = [
    'DUMPS_CREDENTIALS',     # LSASS / SAM memory access
    'INJECTS_INTO_PROCESS',  # remote thread / handle abuse
    'LOADS_SUSPICIOUS_DLL',  # dbghelp/dbgcore etc.
    'MODIFIES_REGISTRY',     # persistence / config
    'SPAWNS_PROCESS',        # process lineage
]
# extend, de-duplicated
for r in HOST_RELATIONS:
    if r not in SECURITY_RELATIONS:
        SECURITY_RELATIONS.append(r)

RELATION_GUIDE += """
  DUMPS_CREDENTIALS        : accessing LSASS/SAM process memory to extract credentials
  INJECTS_INTO_PROCESS     : opening a handle to another process to inject or read memory
  LOADS_SUSPICIOUS_DLL     : loading libraries commonly abused for memory manipulation
  MODIFIES_REGISTRY        : creating or setting registry values
  SPAWNS_PROCESS           : a parent process creating a child process
""".rstrip()

# make the ground-truth technique reachable from extracted relations
RELATION_TO_TECHNIQUES.update({
    'DUMPS_CREDENTIALS':    ['T1003.001', 'T1003', 'T1555'],
    'INJECTS_INTO_PROCESS': ['T1055', 'T1055.001'],
    'LOADS_SUSPICIOUS_DLL': ['T1574', 'T1055'],
    'MODIFIES_REGISTRY':    ['T1112', 'T1547.001'],
    'SPAWNS_PROCESS':       ['T1059', 'T1106'],
})
RELATION_TO_TACTIC_SCOPE.update({
    'DUMPS_CREDENTIALS':    'credential-access',
    'INJECTS_INTO_PROCESS': 'defense-evasion',
    'LOADS_SUSPICIOUS_DLL': 'defense-evasion',
    'MODIFIES_REGISTRY':    'defense-evasion',
    'SPAWNS_PROCESS':       'execution',
})

# rebuild grammar with the extended enum
from llama_cpp import LlamaGrammar
TRIPLE_SCHEMA['properties']['triples']['items']['properties']['relation']['enum'] = SECURITY_RELATIONS
grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))
print(f"Grammar rebuilt — {len(SECURITY_RELATIONS)} relations "
      f"({len(HOST_RELATIONS)} host-specific added)")

In [ ]:
# =====================================================================
#  CELL C — HOST-ADAPTED TRIPLE EXTRACTION
#  ---------------------------------------------------------------------
#  Same signature extract_triples(alert_texts, community_id) and same
#  grammar/LLM as CIC-IDS. Only the prompt examples change: host behaviors
#  (process->dumps->lsass) instead of netflow (scanner->scans->port).
#  Keeps the label-leakage assert from your original.
# =====================================================================
def extract_triples(alert_texts: list, community_id: int) -> list:
    if 'llm' not in globals():
        raise RuntimeError('LLM not initialized. Run the `llm = Llama(...)` cell first.')
    block = '\n'.join(f'- {t}' for t in alert_texts[:6])
    assert ' Label' not in block and 'Technique' not in block, \
        f"Potential label leakage in community {community_id}"

    prompt = f"""[INST] You are a cybersecurity analyst performing threat analysis.
Read the host telemetry alerts below and extract between 1 and 4 semantic triples
describing the attack behaviour using standard security terminology.

{RELATION_GUIDE}

For each triple:
- subject : name the specific process or actor observed (e.g. 'dumpert_exe', 'svchost_proc')
- relation: choose the ONE relation that best fits — use process lineage, memory access,
            granted-access rights, loaded modules, and registry activity to decide
- target  : name the specific resource (e.g. 'lsass_memory', 'run_key', 'remote_process')

Name what you observe. No generic placeholders like 'process' or 'target'.

Example 1 (LSASS credential dumping):
{{"triples": [
  {{"subject": "dumpert_exe",   "relation": "DUMPS_CREDENTIALS",    "target": "lsass_memory"}},
  {{"subject": "dumpert_exe",   "relation": "LOADS_SUSPICIOUS_DLL", "target": "dbghelp_dll"}},
  {{"subject": "cmd_exe",       "relation": "SPAWNS_PROCESS",       "target": "dumpert_exe"}}
]}}

Example 2 (process injection):
{{"triples": [
  {{"subject": "malicious_proc", "relation": "INJECTS_INTO_PROCESS", "target": "explorer_exe"}},
  {{"subject": "malicious_proc", "relation": "EXECUTES_PAYLOAD",     "target": "shellcode_module"}}
]}}

ALERTS (Community {community_id}):
{block}

Return ONLY the JSON object. [/INST]"""

    try:
        out = llm(prompt, grammar=grammar, max_tokens=512, temperature=0.0)
        raw = out['choices'][0]['text'].strip()
        parsed = json.loads(raw)
        triples = parsed.get('triples', [])
        # normalise entity names exactly as CIC-IDS did
        for t in triples:
            t['subject'] = normalise_entity(t.get('subject', ''))
            t['target']  = normalise_entity(t.get('target', ''))
        return triples
    except Exception as e:
        print(f'  [Community {community_id}] parse failed: {e}')
        return []

In [ ]:
# =====================================================================
#  CELL D — RUN EXTRACTION OVER COMMUNITIES
#  ---------------------------------------------------------------------
#  Same structure as CIC-IDS cell 29: group by community_id, extract,
#  store in community_triples keyed by str(cid). MIN size = 1 here because
#  singleton communities are expected at atomic scale.
# =====================================================================
from pathlib import Path
community_df = alerts[alerts['community_id'] != -1].copy()
community_triples = {}
for cid, group in community_df.groupby('community_id'):
    texts = group['semantic_alert'].tolist()
    triples = extract_triples(texts, int(cid))
    community_triples[str(cid)] = triples
    rels = [t['relation'] for t in triples]
    print(f"  Community {cid}: {len(triples)} triples  {rels}")

# persist (mirrors your RESULTS_DIR convention)
try:
    with open(Path(RESULTS_DIR) / 'host_community_triples.json', 'w', encoding='utf-8') as f:
        json.dump(community_triples, f, indent=2)
except Exception:
    pass  # RESULTS_DIR may not be defined in a fresh kernel

In [ ]:
# =====================================================================
#  CELL E — KNOWLEDGE GRAPH  (reuses your two-layer design)
#  ---------------------------------------------------------------------
#  Identical construction to CIC-IDS cell 32: behavioral layer from triples,
#  ATT&CK layer from attck_techniques, bridge edges via RELATION_TO_TECHNIQUES.
# =====================================================================
import networkx as nx
G = nx.DiGraph()
edge_weights, edge_communities = {}, {}
total_triples = invalid_triples = 0

for cid, triples in community_triples.items():
    for t in triples:
        s, r, o = (str(t.get(k, '')).strip() for k in ('subject', 'relation', 'target'))
        if not (s and r and o):
            invalid_triples += 1
            continue
        total_triples += 1
        key = (s, r, o)
        edge_weights[key] = edge_weights.get(key, 0) + 1
        edge_communities.setdefault(key, []).append(cid)

for (src, rel, tgt), w in edge_weights.items():
    G.add_node(src, layer='behavioral')
    G.add_node(tgt, layer='behavioral')
    G.add_edge(src, tgt, relation=rel, weight=w, layer='behavioral',
               communities=','.join(edge_communities[(src, rel, tgt)]))

if 'attck_techniques' not in globals():
    raise RuntimeError('Run the ATT&CK load cell first.')
for tid, info in attck_techniques.items():
    if tid not in G:
        G.add_node(tid, layer='attck', name=info.get('name', ''),
                   tactic=(info.get('tactics') or [''])[0],
                   description=info.get('description', ''))

# bridge behavioral relations -> candidate techniques
bridges = 0
for (src, rel, tgt) in edge_weights:
    for tid in RELATION_TO_TECHNIQUES.get(rel, []):
        if tid in G:
            G.add_edge(tgt, tid, relation='MAPS_TO', layer='bridge')
            bridges += 1
print(f"KG: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges | "
      f"{total_triples} triples ({invalid_triples} invalid), {bridges} bridges")

In [ ]:
# =====================================================================
#  CELL F — EVALUATE: does the pipeline recover T1003.001?
#  ---------------------------------------------------------------------
#  Mirrors your parent_match + kg_candidate_tids evaluation. For each
#  community, the predicted technique set comes from the DOMINANT extracted
#  relation -> RELATION_TO_TECHNIQUES. Scored against ground_truth_technique.
# =====================================================================
from collections import Counter

def parent_match(gt_id: str, gen_id: str) -> bool:
    """T1003.001 matches T1003 and vice-versa (sub-technique tolerant)."""
    if not gt_id or not gen_id:
        return False
    return gt_id == gen_id or gt_id.split('.')[0] == gen_id.split('.')[0]

def kg_candidate_tids(cid):
    rels = [t.get('relation', '') for t in community_triples.get(str(cid), [])]
    if not rels:
        return []
    dominant = Counter(rels).most_common(1)[0][0]
    return RELATION_TO_TECHNIQUES.get(dominant, [])

rows = []
for cid in sorted(community_df['community_id'].unique()):
    g = community_df[community_df['community_id'] == cid]
    gt = g['ground_truth_technique'].mode()
    gt = gt.iloc[0] if len(gt) else None
    cands = kg_candidate_tids(cid)
    hit = any(parent_match(gt, c) for c in cands)
    rows.append({'community_id': cid, 'ground_truth': gt,
                 'candidates': cands[:3], 'hit': hit,
                 'image': g['image'].iloc[0]})

eval_df = pd.DataFrame(rows)
acc = eval_df['hit'].mean() if len(eval_df) else 0.0
print(eval_df.to_string(index=False))
print(f"\nKG technique-recovery accuracy: {acc:.0%} "
      f"({eval_df['hit'].sum()}/{len(eval_df)} communities)")

# -------------------------------------------------------------------
#  IMPORTANT — how to read this number for an ATOMIC dataset.
#  This capture has ONE dataset-level label (T1003.001) applied to EVERY
#  process, including benign system processes (Explorer, csrss, cmd...).
#  Those benign processes correctly do NOT map to credential dumping, but
#  they count as "misses" against the blanket label. So overall accuracy
#  UNDERSTATES pipeline quality and should NOT be reported as a headline.
#
#  The meaningful prototype question is: did the pipeline identify the
#  MALICIOUS process and map it to the correct technique?
# -------------------------------------------------------------------
mal = eval_df[eval_df['image'].str.contains('Dumpert', case=False)]
if len(mal):
    print(f"\nMalicious-process check (Outflank-Dumpert.exe): "
          f"{'CORRECTLY mapped to T1003.001' if mal['hit'].all() else 'MISSED'}")
print("Note: benign-process scores are a triage/anomaly question, not "
      "technique-classification. Report the malicious-process result for "
      "the atomic prototype; report macro-accuracy only at APT29 scale "
      "where multiple distinct ground-truth techniques exist.")

In [ ]:
# =====================================================================
#  CELL G — (OPTIONAL) RAG REPORT for the malicious community
#  ---------------------------------------------------------------------
#  Reuses your ChromaDB collection + generate_rag_report. Only run after
#  CELL 35's ChromaDB build has executed in this kernel. This confirms the
#  end-to-end RAG path produces a grounded report on host data.
# =====================================================================
def run_rag_demo():
    if 'collection' not in globals():
        print("Run the ChromaDB build cell (CIC-IDS cell 35) first.")
        return
    # pick the community whose dominant relation is credential dumping
    target_cid = None
    for cid in community_df['community_id'].unique():
        rels = [t.get('relation','') for t in community_triples.get(str(cid), [])]
        if 'DUMPS_CREDENTIALS' in rels:
            target_cid = cid
            break
    if target_cid is None:
        print("No credential-dumping community found — check extraction output.")
        return
    scope = 'credential-access'
    q = "process accessing LSASS memory to extract credentials"
    res = collection.query(query_texts=[q], n_results=5,
                           where={'tactic': scope})
    print(f"Community {target_cid} — top ChromaDB hits (scope={scope}):")
    for tid, doc in zip(res['ids'][0], res['documents'][0]):
        print(f"  {tid}: {doc.splitlines()[1]}")  # the Name line

run_rag_demo()